# Heitler-London and Hund-Mulliken (HL/HM) exchange approximations

To approximate exchange interaction between a pair of quantum dots, `qudipy` fits the potential 
landscape to a (biased) quartic potential and use formulas for HL/HM interaction
from [Burkard et al, PRB, 1999]( https://journals.aps.org/prb/abstract/10.1103/PhysRevB.59.2070).

The tutorial present the dependencies of exchange on gate voltages and confirm their exponential nature. 

## Importing relevant modules

In [ ]:
import os, sys
sys.path.append(os.path.dirname(os.getcwd()))

import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family']='sans-serif'

# Potential module.
import qudipy.potential as pot
import qudipy.potential.manipulate as manip

# Control shapes and voltage pulse routines -- will be needed for later
from qudipy.control import shapes, voltage_pulses, balance_zeeman

# Constants class.
from qudipy.utils.constants import Constants

#Exchange determination routines
import qudipy.exchange.exchange as exch

## Importing nextnano simulation data 

The test case is a 2-dot system defined by a pair of plunger gates 
and a tunneling gate in between.  
This tutorial uses the pre-processed data from SharePoint. 


In [ ]:
# defining system constants:
csts = Constants('Si/SiO2')

# Define a directory location which will contain the pre-processed potential
# data for a given 2D slice. 
output_preprocessed = os.path.join(sys.path[0], 'QuDiPy tutorial data', 
                                                'Pre-processed potentials',
                                                     'Pre-processed_data')

Defining the slice and importing all data

In [ ]:
z = -0.2
pot_dir = output_preprocessed + '_for_nearest_slice{:.3e}'.format(z)

ctrl_vals = [[0.0], 
                [0.2, 0.25, 0.3, 0.35, 0.4, 0.45], 
                [-0.1, -0.05, 0.0, 0.05, 0.1, 0.15], 
                [0.2, 0.25, 0.3, 0.35, 0.4, 0.45], 
                [0.0], [0.0], [0.0]]

ctrl_names = ('GateContact_T1', 'GateContact_P1', 'GateContact_T2', 
        'GateContact_P2', 'GateContact_T3', 'GateContact_S', 'SiContact')

Importing potential and electric field data

In [ ]:
pot_data = pot.load_potentials(ctrl_vals, ctrl_names,
                                  f_type='pot', f_dir=pot_dir,
                                  f_dis_units='nm', f_pot_units='V')

e_field_data = pot.load_potentials(ctrl_vals, ctrl_names,
                                  f_type='pot', f_dir=pot_dir,
                                  f_dis_units='nm')

Building interpolators for convenient plotting: 

In [ ]:
pot_interp = pot.build_interpolator(pot_data, constants=csts)
e_interp = pot.build_interpolator(e_field_data, constants=csts)

## HL exchange calculations

### Comparing fitted potentials

Building GridParameters object first

In [ ]:
gparams = pot.GridParameters(*pot_data['coords'])

Constructing 2D potentials and getting their fits

In [ ]:
pot1 = pot_interp([0.3267, 0.14, 0.241])
gparams.update_potential(pot1)
fit1 = manip.fit_quartic(gparams, material='Si/SiO2', return_params=True)

X, Y = gparams.x_mesh *1e9, gparams.y_mesh *1e9 # converted in nm

[print(f'{key}: ', fit1[key]) for key in 
            ['dot_sep', 'e_field_x', 'omega_0', 'x_centre', 'U_0']];

In [ ]:
plt.figure(dpi=100)
plt.pcolormesh(X, Y, pot1, cmap='viridis')
plt.colorbar()
plt.show()

Plotting nextnano-generated potential together with the quartic function fit in 3D

In [ ]:
fig = plt.figure( dpi=200)
ax = plt.axes(projection='3d')

# limiting max height
offcut = 0.6
pot_max = offcut * pot1.min() + (1- offcut) * pot1.max()

ax.contour(X, Y, np.minimum(pot_max, pot1) /1.6e-19, 10, cmap='magma')
ax.contour(X, Y, np.minimum(pot_max, fit1['U_fit'])  /1.6e-19, 30, cmap='spring')

ax.tick_params(labelsize=5)

ax.set_xlabel('xcoords [nm]')
ax.set_ylabel('ycoords [nm]')
ax.set_zlabel('U [eV]')

ax.set_xlim(-40, 40)
ax.set_ylim(-30,30)
# ax.set_zlim(-2.23, -2.20)
plt.show()

### Sweeping tunneling gate

Calculating exchange values for different tunneling gate voltages. 
Values of plunger gates are fixed.

In [ ]:
# fixed plunger gate voltages 
V_P1 = V_P2 = 0.2

# voltage values on tunneling gate
num = 30
V_tuns = np.linspace(0, 0.135, num)

#calculating exchange
J_hl_tun = np.zeros(num)
J_hm_tun = np.zeros(num)
for idx in range(num): 
    gparams.update_potential(pot_interp([V_P1, V_tuns[idx], V_P2]))
    J_hl_tun[idx] = exch.hl_quartic(gparams, material='Si/SiO2')
    J_hm_tun[idx] = exch.hm_quartic(gparams, material='Si/SiO2')

Plotting HM and HL exchange versus tunneling gate:

In [ ]:
fig, ax = plt.subplots(1, figsize=(6,4), dpi=120)

ax.plot(V_tuns, J_hl_tun / 1.6e-25, '--',
                    label = 'HL, quartic fit', linewidth=2)
ax.plot(V_tuns, J_hm_tun / 1.6e-25, 'tab:red',
                    label = 'HM, quartic fit', linewidth=2)
ax.set_yscale('log')

# setting ticks
ax.set_xticks([0,0.04,0.08,0.12])
ax.set_yticks([1,1e-4, 1e-8])
ax.tick_params(labelsize=14)

ax.set_xlabel('Tunneling gate voltage, V', fontsize=18)
ax.set_ylabel('J, $\mu$eV', fontsize=18)
ax.legend(loc='upper left', fontsize=18)

ax.text((V_tuns[0]+V_tuns[-1])/2, J_hm_tun[0]/ 1.6e-25, 
'V$_{1}$ = ' + str(V_P1) + ' V\nV$_{2}$ = ' + str(V_P2) 
                    +  ' V', fontsize=18,
                            bbox=dict(facecolor='blue', alpha=0.2))


plt.show()

### Sweeping bias
Calculating exchange for different biases: either symmetric or asymmetric, when 
only one plunger gate voltage changes.
Tunneling gate voltage remains fixed.

In [ ]:
# initial plunger gate voltages (no bias)

# V_P0 = 0.27   # uncomment for the *symmetric* bias sweep
V_P0 = 0.21     # uncomment for the *asymmetric* bias sweep

# fixed tunneling gate voltage
V_tun = 0.1

# biases
num = 120
V_biases_sym = np.linspace(-0.12, 0.12, num)        # symmetric bias
dV_Ps = np.linspace(0, 0.12, num)                    # asymmetric bias

#calculating exchange
J_hl_bias = np.zeros(num)
J_hm_bias = np.zeros(num)
for idx in range(num): 
    # **uncomment either symmetric (1st) or asymmetric (2nd) bias sweep  below**

    # gparams.update_potential(pot_interp([V_P0 + V_biases_sym[idx]/2, V_tun, V_P0 - V_biases_sym[idx]/2]))
    gparams.update_potential(pot_interp([V_P0, V_tun, V_P0 + dV_Ps[idx]]))

    J_hl_bias[idx] = exch.hl_quartic(gparams, material='Si/SiO2')
    J_hm_bias[idx] = exch.hm_quartic(gparams, material='Si/SiO2')


Plotting HL and HM exchange versus bias

In [ ]:
fig, ax = plt.subplots(1, figsize=(5,3), dpi=150)
# **uncomment either symmetric (1st) or asymmetric (2nd) bias sweep  below**

# ax.plot(V_biases_sym, J_hl_bias / 1.6e-25, label = 'HL, quartic fit')      #exchange in ueV
# ax.plot(V_biases_sym, J_hm_bias / 1.6e-25, label = 'HM, quartic fit')

ax.plot(dV_Ps, J_hl_bias / 1.6e-25, label = 'HL, quartic fit')      #exchange in ueV
ax.plot(dV_Ps, J_hm_bias / 1.6e-25, label = 'HM, quartic fit')

ax.set_yscale('log')
ax.set_xlabel('V$_{bias}$, V', fontsize=12)
ax.set_ylabel('J, $\mu$eV', fontsize=12)
ax.legend(loc='best', bbox_to_anchor=(0.5, 0.7))

plt.show()